# Colab 后端启动 (FastAPI + SDXL-Turbo + cloudflared 公网隧道)

**用途**：把这个项目的后端跑在 Colab 的 GPU 上，给你一个公网 URL，本地前端直接连过来用。

## 使用步骤

1. 顶部菜单 `代码执行程序 → 更改运行时类型 → 硬件加速器：T4 GPU → 保存`
2. 把下面 **第 1 格** 的 `REPO_URL` 改成你 GitHub 仓库地址
3. `代码执行程序 → 全部运行`，等首次模型下载（~7GB，约 3-5 分钟）
4. **第 3 格** 跑完会打印一个 `https://xxx.trycloudflare.com` 公网地址
5. 把它复制到本地前端的 `image-corrector-ui/.env.local`：
   ```
   VITE_API_BASE_URL=https://xxx.trycloudflare.com/api
   ```
6. 本地重启前端 `npm run dev`，刷新浏览器，所有处理请求就走 Colab GPU 了

**注意**：不要关闭这个 Colab 标签页，否则隧道断开。Colab 免费版连续运行约 12 小时后会自动断。

In [ ]:
# === 1. Clone GitHub 仓库 ===
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO"  # ← 改成你的仓库地址
BRANCH = "main"

import os, shutil
if os.path.exists("/content/dip_project"):
    shutil.rmtree("/content/dip_project")

!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/dip_project
!ls /content/dip_project/image-processing-backend

In [ ]:
# === 2. 安装依赖 ===
# Colab 已自带 GPU torch，所以这里只补 requirements.txt + requirements-ai.txt
%cd /content/dip_project/image-processing-backend
!pip install -q -r requirements.txt
!pip install -q -r requirements-ai.txt

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# === 3. 下载 cloudflared + 启动后端 + 起隧道 ===
import subprocess, time, re, os, sys, signal

# 下载 cloudflared 二进制（无需登录的免费 quick tunnel）
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 关掉可能残留的进程
!pkill -f 'python main.py' 2>/dev/null; pkill -f cloudflared 2>/dev/null; true
time.sleep(1)

# 启动 FastAPI（关闭 reload，避免 Colab 信号问题）
backend_log = open("/content/backend.log", "w")
backend = subprocess.Popen(
    ["python", "-c",
     "import uvicorn; from main import app; "
     "uvicorn.run(app, host='127.0.0.1', port=8000, log_level='info')"],
    cwd="/content/dip_project/image-processing-backend",
    stdout=backend_log, stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
print("后端启动中...")

# 等后端 ready
import urllib.request, urllib.error
for i in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/docs", timeout=1)
        print("✅ 后端就绪 (http://127.0.0.1:8000)")
        break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ 后端启动超时，查看日志：")
    !tail -50 /content/backend.log
    raise RuntimeError("backend failed")

# 启动 cloudflared quick tunnel
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
url_re = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
for _ in range(120):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2); continue
    print(line.rstrip())
    m = url_re.search(line)
    if m:
        public_url = m.group(0)
        break

print()
print("=" * 64)
if public_url:
    print("🎉 公网 URL:", public_url)
    print()
    print("在本地仓库的 image-corrector-ui/.env.local 写入：")
    print(f"  VITE_API_BASE_URL={public_url}/api")
    print()
    print("然后本地重启 npm run dev，刷新浏览器即可。")
else:
    print("❌ 没拿到 cloudflared 公网 URL，请重跑本格。")
print("=" * 64)

In [ ]:
# === 4. 保持运行 (不要关闭这个标签页) ===
# 这一格会持续打印后端日志。需要停止时点单元格旁的停止按钮。
import time, os
try:
    last_size = 0
    while True:
        time.sleep(2)
        if os.path.exists("/content/backend.log"):
            size = os.path.getsize("/content/backend.log")
            if size > last_size:
                with open("/content/backend.log") as f:
                    f.seek(last_size)
                    chunk = f.read()
                    print(chunk, end="")
                last_size = size
except KeyboardInterrupt:
    print("\n停止隧道...")
    !pkill -f cloudflared
    !pkill -f 'main:app'